In [ ]:
import os
import math
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from utils import build_ensemble_data_list
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix

In [ ]:
SEED = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4
betas=(0.9, 0.999)
eps=1e-8
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
PATIENCE = 30
MODEL_SAVE_PATH = "./EvoStruct-Kla/best_model.pt"
NUM_CLASSES = None

In [ ]:
class Contact_MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, attn_dropout=0.1):
        super(Contact_MultiHeadAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        assert hidden_dim % num_heads == 0,
        self.head_dim = hidden_dim // num_heads

        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)

        self.eps = 1e-6
        self.log_lambda = nn.Parameter(torch.tensor(-2.0))

        self.attn_dropout = nn.Dropout(attn_dropout)

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, esm_contact, mask):

        B, N, d = x.shape


        Q = self.W_q(x)  # [B, N, d]
        K = self.W_k(x)  # [B, N, d]
        V = self.W_v(x)  # [B, N, d]

        Q = Q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d_h]
        K = K.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d_h]
        V = V.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d_h]

        scores = torch.matmul(Q, K.transpose(-2, -1))  # [B, h, N, N]

        mask_2d = mask.unsqueeze(1) * mask.unsqueeze(2)  # [B, N, N]
        esm_contact = esm_contact.clamp(min=0.0, max=1.0)
        esm_contact = esm_contact.unsqueeze(1)

        lambda_ = F.softplus(self.log_lambda)
        bias = torch.clamp(torch.log(esm_contact + self.eps), min =-5.0)
        bias = bias - bias.mean(dim=-1, keepdim=True)
        scores_content = scores / math.sqrt(self.head_dim)
        bias_term = lambda_ * bias
        scores = scores_content + bias_term
        scores = scores.masked_fill(mask_2d.unsqueeze(1) == 0, -1e9)

        # softmax
        attn = F.softmax(scores, dim=-1)  # [B, h, N, N]
        attn = self.attn_dropout(attn)

        attn_out = torch.matmul(attn, V)  # [B, h, N, d_h]

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, N, self.hidden_dim)

        attn_out = self.out_proj(attn_out)  # [B, N, d]
        attn_out = attn_out * mask.unsqueeze(-1).to(attn_out.dtype)

        return attn_out



class Contact_Attention_Block(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1, attn_dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn = Contact_MultiHeadAttention(hidden_dim, num_heads, attn_dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, contacts, mask):
        # Attention + residual
        x = x + self.dropout1(
            self.attn(self.norm1(x), contacts, mask)
        )
        # FFN + residual
        x = x + self.dropout2(
            self.ffn(self.norm2(x))
        )
        return x



class Contact_Attention(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_heads, dropout=0.1, attn_dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(embedding_dim, hidden_dim)
        self.attention_block1 = Contact_Attention_Block(hidden_dim, num_heads, dropout, attn_dropout)
        self.attention_block2 = Contact_Attention_Block(hidden_dim, num_heads, dropout, attn_dropout)

    def forward(self, x, contacts, mask):
        out = self.linear1(x)
        out = self.attention_block1(out, contacts, mask)
        out = self.attention_block2(out, contacts, mask)

        return out



class attn_model(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_heads=4, dropout=0.1, attn_dropout=0.1):
        super().__init__()
        self.contact_attention = Contact_Attention(embed_dim, hidden_dim, num_heads, dropout, attn_dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 1)
        )

        prior = 0.421711541759561
        init_bias = math.log(prior/(1-prior))
        with torch.no_grad():
            self.classifier[-1].bias.fill_(init_bias)



    def forward(self, x, contacts, mask):
        out = self.contact_attention(x, contacts, mask)

        mask_f = mask.unsqueeze(-1).to(out.dtype)              # [B,N,1]
        K_feat = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(K_feat).squeeze(-1)  # [B]
        return logits

In [ ]:
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels=512, num_layers=2, dropout=0.3,
                 heads=4, gat_concat=True, att_dropout=0.0, L=35, center_local=17):
        super().__init__()
        self.input_lin = nn.Linear(in_channels, hidden_channels)
        self.num_layers = num_layers
        self.dropout = dropout
        self.L = L
        self.center_local = center_local

        out_dim = hidden_channels
        self.convs = nn.ModuleList()
        for i in range(num_layers):
            if gat_concat:
                if hidden_channels % heads == 0:
                    out_per_head = hidden_channels // heads
                else:
                    out_per_head = hidden_channels
                conv = GATv2Conv(out_dim, out_per_head, heads=heads, concat=True, dropout=att_dropout)
                out_dim = out_per_head * heads
            else:
                conv = GATv2Conv(out_dim, hidden_channels, heads=heads, concat=False, dropout=att_dropout)
                out_dim = hidden_channels
            self.convs.append(conv)

        self.project = nn.Linear(out_dim, hidden_channels) if out_dim != hidden_channels else None

        self.mlp = nn.Sequential(
            nn.LayerNorm(hidden_channels),
            nn.Linear(hidden_channels, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, batch, node_mask):
        x = self.input_lin(x)
        layer_features = []

        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x * node_mask.unsqueeze(-1).float()
            layer_features.append(x)

        node_repr = torch.stack(layer_features, dim=0).mean(dim=0)

        if self.project is not None:
            node_repr = self.project(node_repr)

        batch_size = int(batch.max().item() + 1)
        device = node_repr.device
        graph_idx = torch.arange(batch_size, device=device)
        center_global_idx = graph_idx * self.L + self.center_local
        center_feats = node_repr[center_global_idx]

        out = self.mlp(center_feats)
        return out.squeeze(1)


In [ ]:
# Z-logits Ensemble
class ensemble_model(nn.Module):
    def __init__(self, GAT_in_channels, Atten_embed_dim, Atten_hidden_dim,
                 GAT_hidden_channels=512, GAT_num_layers=2, GAT_dropout=0.3,
                 GAT_heads=4, GAT_gat_concat=True, GAT_att_dropout=0.0, GAT_L=35, GAT_center_local=17,
                 Atten_num_heads=4, Atten_dropout=0.1, Atten_attn_dropout=0.1):
        super().__init__()
        self.gat_model = GAT(GAT_in_channels, GAT_hidden_channels, GAT_num_layers, GAT_dropout,
                             GAT_heads, GAT_gat_concat, GAT_att_dropout, GAT_L, GAT_center_local)
        self.atten_model = attn_model(Atten_embed_dim, Atten_hidden_dim, Atten_num_heads, Atten_dropout, Atten_attn_dropout)

        self.register_buffer("g_mean", torch.tensor(-0.31586003))
        self.register_buffer("g_std",  torch.tensor( 1.8720187))
        self.register_buffer("a_mean", torch.tensor(-0.35089943))
        self.register_buffer("a_std",  torch.tensor( 0.84683055))

        self.log_s_g = nn.Parameter(torch.tensor(0.0))  # scale_g = exp(log_s_g)
        self.log_s_a = nn.Parameter(torch.tensor(0.0))

    def forward(self, gat_x, gat_edge_index, gat_batch, gat_node_mask,
                atten_x, atten_contacts, atten_mask):
        g = self.gat_model(gat_x, gat_edge_index, gat_batch, gat_node_mask).view(-1)
        a = self.atten_model(atten_x, atten_contacts, atten_mask).view(-1)

        sg = torch.exp(self.log_s_g).clamp(1e-3, 1e3)
        sa = torch.exp(self.log_s_a).clamp(1e-3, 1e3)

        g = (g - self.g_mean) / (self.g_std + 1e-6)
        a = (a - self.a_mean) / (self.a_std + 1e-6)

        combined_logits = sg * g + sa * a
        return combined_logits


In [ ]:
def set_global_seed(seed: int = 12, deterministic: bool = True):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    if deterministic:
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

set_global_seed(SEED)

def find_best_threshold(probs, targets, metric='f1', thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)
    best_t, best_score = 0.5, -1.0
    rows = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        if metric == 'f1':
            score = f1_score(targets, preds, zero_division=0)
        elif metric == 'mcc':
            score = matthews_corrcoef(targets, preds)
        else:
            raise ValueError("metric must be 'f1' or 'mcc'")
        tn, fp, fn, tp = confusion_matrix(targets, preds).ravel()
        rows.append((t, score, tp, tn, fp, fn))
        if score > best_score:
            best_score, best_t = score, t
    return best_t, best_score, rows

def infer_num_classes_and_in_dim(data_list):
    in_dim = None
    labels = []
    for d in data_list:
        if in_dim is None and getattr(d, "x", None) is not None:
            in_dim = d.x.size(1)
        y = d.y
        if isinstance(y, torch.Tensor):
            labels.append(int(y.view(-1)[0].item()))
        else:
            labels.append(int(y))
    if in_dim is None:
        raise ValueError("Cannot infer input feature dimension from data_list")
    num_classes = len(set(labels))
    return in_dim, num_classes

def worker_init_fn(worker_id):
    seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(seed)
    random.seed(seed)

def to_loader(data_list, batch_size, shuffle=True, num_workers=0, seed=SEED):
    if shuffle:
        g = torch.Generator()
        g.manual_seed(seed)
    else:
        g = None
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers,
                      worker_init_fn=worker_init_fn if num_workers > 0 else None,
                      generator=g)

def confusion_stats(preds: torch.Tensor, targets: torch.Tensor):
    preds_np = preds.cpu().numpy().astype(int)
    targets_np = targets.cpu().numpy().astype(int)
    TP = int(((preds_np == 1) & (targets_np == 1)).sum())
    TN = int(((preds_np == 0) & (targets_np == 0)).sum())
    FP = int(((preds_np == 1) & (targets_np == 0)).sum())
    FN = int(((preds_np == 0) & (targets_np == 1)).sum())
    return TP, TN, FP, FN

def compute_metrics_from_confusion(TP: int, TN: int, FP: int, FN: int):
    total = TP + TN + FP + FN
    acc = (TP + TN) / total if total > 0 else float("nan")

    recall = TP / (TP + FN) if (TP + FN) > 0 else float("nan")   # SEN / Sn / Re
    specificity = TN / (TN + FP) if (TN + FP) > 0 else float("nan")  # Sp
    precision = TP / (TP + FP) if (TP + FP) > 0 else float("nan")    # Pr

    if (precision is float("nan")) or (recall is float("nan")):
        f1 = float("nan")
    elif (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    denom = (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)
    if denom <= 0:
        mcc = float("nan")
    else:
        mcc = (TP * TN - FP * FN) / math.sqrt(denom)

    return {
        "acc": acc,
        "recall": recall,
        "specificity": specificity,
        "precision": precision,
        "f1": f1,
        "mcc": mcc
    }

def train_epoch(model, loader, optimizer, criterion, device, scheduler=None):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # out = model(batch.x, batch.edge_index, batch.batch, batch.node_mask)  # expected shape: [B]
        out = model(batch.x, batch.edge_index, batch.batch, batch.node_mask, batch.x_seq, batch.esm_contact, batch.seq_mask)  # logits [B]

        y = batch.y.view(-1).float().to(device)             # shape [B], dtype float

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        batch_size = out.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        probs = torch.sigmoid(out)                          # shape [B]
        preds = (probs > 0.5).long()                        # shape [B], dtype long

        all_preds.append(preds.cpu())
        all_targets.append(batch.y.view(-1).long().cpu())

    preds_all = torch.cat(all_preds, dim=0)
    targets_all = torch.cat(all_targets, dim=0)
    TP, TN, FP, FN = confusion_stats(preds_all, targets_all)
    metrics = compute_metrics_from_confusion(TP, TN, FP, FN)
    avg_loss = total_loss / total_examples if total_examples > 0 else float("nan")
    return avg_loss, metrics, preds_all, targets_all

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_probs = []
    all_preds = []
    all_targets = []

    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch, batch.node_mask, batch.x_seq, batch.esm_contact, batch.seq_mask)  # logits [B]

        y = batch.y.view(-1).float().to(device)              # [B], float
        loss = criterion(out, y)

        batch_size = out.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        probs = torch.sigmoid(out).cpu()                      # [B], cpu tensor of probabilities
        preds = (probs > 0.5).long()                          # [B], cpu tensor of binarized preds

        all_probs.append(probs)
        all_preds.append(preds.cpu())
        all_targets.append(batch.y.view(-1).long().cpu())

    probs_all = torch.cat(all_probs, dim=0).numpy()         # numpy array shape [N]
    preds_all = torch.cat(all_preds, dim=0)
    targets_all = torch.cat(all_targets, dim=0).numpy()     # numpy array shape [N]

    TP, TN, FP, FN = confusion_stats(preds_all, torch.from_numpy(targets_all))
    metrics = compute_metrics_from_confusion(TP, TN, FP, FN)

    avg_loss = total_loss / total_examples if total_examples > 0 else float("nan")
    return avg_loss, metrics, probs_all, targets_all


def run_training(train_list, val_list, test_list, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, betas=betas, eps=eps):

    in_dim, num_classes = infer_num_classes_and_in_dim(train_list + val_list + test_list) if NUM_CLASSES is None else (None, NUM_CLASSES)
    if NUM_CLASSES is None:
        print(f"Inferred in_dim={in_dim}, num_classes={num_classes}")
    else:
        num_classes = NUM_CLASSES
        in_dim = in_dim or infer_num_classes_and_in_dim(train_list + val_list + test_list)[0]
        print(f"Using provided num_classes={num_classes}, in_dim={in_dim}")

    model = ensemble_model(GAT_in_channels=in_dim, Atten_embed_dim=in_dim, Atten_hidden_dim=512)
    model = model.to(DEVICE)

    train_loader = to_loader(train_list, batch_size=batch_size, shuffle=True, num_workers=0, seed=SEED)
    val_loader = to_loader(val_list, batch_size=batch_size, shuffle=False, num_workers=0, seed=SEED+1) if len(val_list) > 0 else None
    test_loader = to_loader(test_list, batch_size=batch_size, shuffle=False, num_workers=0, seed=SEED+2) if len(test_list) > 0 else None

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=betas, eps=eps)

    steps_per_epoch = max(1, len(train_loader))
    total_steps = epochs * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=total_steps, pct_start=0.1, anneal_strategy='cos', final_div_factor=1e3)
    best_val_mcc = -float("inf")
    best_val_acc = -1.0
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_metrics, _, _ = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scheduler)
        val_loss, val_metrics, _, _ = evaluate(model, val_loader, criterion, DEVICE)


        def fmt(metrics):
            return (f"acc {metrics['acc']:.4f} sn {metrics['recall']:.4f} sp {metrics['specificity']:.4f} "
                    f"pr {metrics['precision']:.4f} f1 {metrics['f1']:.4f} mcc {metrics['mcc'] if not np.isnan(metrics['mcc']) else float('nan'):.4f}")

        print(f"Epoch {epoch:03d} | train_loss {train_loss:.4f} | train_{fmt(train_metrics)} | val_loss {val_loss:.4f} | val_{fmt(val_metrics)}")

        current_val_mcc = val_metrics["mcc"] if not np.isnan(val_metrics["mcc"]) else -float("inf")
        current_val_acc = val_metrics["acc"]

        improved = False
        if current_val_mcc > best_val_mcc:
            improved = True
        elif current_val_mcc == best_val_mcc and current_val_acc > best_val_acc:
            improved = True

        if improved:
            best_val_mcc = current_val_mcc
            best_val_acc = current_val_acc
            epochs_no_improve = 0
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_metrics": val_metrics,
            }, MODEL_SAVE_PATH)
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs, early stopping.")
            break

    ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])


    val_loss, val_metrics, val_probs, val_targets = evaluate(model, val_loader, criterion, DEVICE)
    best_t, best_score, _ = find_best_threshold(val_probs, val_targets, metric="mcc")  # or 'f1'
    print("best threshold on val:", best_t, "score:", best_score)
    if test_loader is not None:
        test_loss, test_metrics, probs_all, targets_all = evaluate(model, test_loader, criterion, DEVICE)
        print("Test results:")
        print(f"  loss {test_loss:.4f}")
        print(f"  acc {test_metrics['acc']:.4f} recall {test_metrics['recall']:.4f} specificity {test_metrics['specificity']:.4f}")
        print(f"  precision {test_metrics['precision']:.4f} f1 {test_metrics['f1']:.4f} mcc {test_metrics['mcc'] if not np.isnan(test_metrics['mcc']) else float('nan'):.4f}")
        return model, (train_loss, train_metrics), (val_loss, val_metrics), (test_loss, test_metrics), probs_all, targets_all
    else:
        return model, (train_loss, train_metrics), (val_loss, val_metrics), None, None, None

In [ ]:
train_x = torch.load("./EvoStruct-Kla/Data/dataset_demo/train/x.pt"))
train_edge_index = torch.load("./EvoStruct-Kla/Data/dataset_demo/train/edge_index.pt")
train_y = torch.load("./EvoStruct-Kla/Data/dataset_demo/train/y.pt")
train_masks = torch.load("./EvoStruct-Kla/Data/dataset_demo/train/mask.pt")
train_esm_contacts = torch.load("./EvoStruct-Kla/Data/dataset_demo/train/esm_contact.pt")

val_x = torch.load("./EvoStruct-Kla/Data/dataset_demo/val/x.pt")
val_edge_index = torch.load("./EvoStruct-Kla/Data/dataset_demo/val/edge_index.pt")
val_y = torch.load("./EvoStruct-Kla/Data/dataset_demo/val/y.pt")
val_masks = torch.load("./EvoStruct-Kla/Data/dataset_demo/val/mask.pt")
val_esm_contacts = torch.load("./EvoStruct-Kla/Data/dataset_demo/val/esm_contact.pt")

test_x = torch.load("./EvoStruct-Kla/Data/dataset_demo/test/x.pt")
test_edge_index = torch.load("./EvoStruct-Kla/Data/dataset_demo/test/edge_index.pt")
test_y = torch.load("./EvoStruct-Kla/Data/dataset_demo/test/y.pt")
test_masks = torch.load("./EvoStruct-Kla/Data/dataset_demo/test/mask.pt")
test_esm_contacts = torch.load("./EvoStruct-Kla/Data/dataset_demo/test/esm_contact.pt")

In [ ]:
if __name__ == "__main__":
    try:
        train_list, val_list, test_list = build_ensemble_data_list(train_x, train_edge_index, train_y, train_masks, train_esm_contacts, train_masks),\
                                        build_ensemble_data_list(val_x, val_edge_index, val_y, val_masks, val_esm_contacts, val_masks),\
                                        build_ensemble_data_list(test_x, test_edge_index, test_y, test_masks, test_esm_contacts, test_masks)
    except Exception:
        raise RuntimeError("Please provide train_list, val_list, test_list or replace import with your lists.")

    model, train_stats, val_stats, test_stats, preds, targets = run_training(train_list, val_list, test_list)